# Day 8: LangGraph — Stateful Agent Workflows 🕸️

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Your first 3-node graph — understand nodes, edges, state
2. A conditional router — agent picks different paths automatically
3. A looping review agent — drafts until quality is met
4. Human-in-the-loop — pause workflow, approve, then resume
5. A supervisor agent — one router delegates to 3 specialists

**Time:** ~2 hours | **Difficulty:** Beginner-friendly (builds step by step) | **Prerequisites:** Days 1–7 complete

---

## 0. Setup — Install LangGraph and Load API Keys

LangGraph is a separate package from LangChain. Run this cell first.

In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment and run if using Google Colab
# LOCAL users: SKIP this cell
# ============================================================

# !pip install langgraph langchain-groq langchain-core python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing — check .env file'}")

# Verify LangGraph is installed
from importlib.metadata import version
print(f"LangGraph version: {version('langgraph')}")

In [ ]:
# We'll reuse this LLM setup throughout the entire notebook
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Quick test
response = llm.invoke("Say 'LangGraph ready!' in exactly 3 words.")
print(response.content)

---
## 1. 🎯 Your First LangGraph — 3 Nodes, Hello World

Before anything complex, let's understand the basic pattern.

We'll build a 3-node graph:
```
[greet] → [process] → [respond]
```

**Remember the 4 steps from the PPT:**
1. Define State
2. Write node functions
3. Build the graph
4. Compile and run

### Step 1 — Define State

In [ ]:
from typing import TypedDict

# State is just a Python TypedDict — a dictionary with type hints
# Every node will read from and write to this shared dictionary

class ConversationState(TypedDict):
    user_name: str      # Input: who is the user?
    greeting: str       # Added by greet_node
    processed: str      # Added by process_node
    final_response: str # Added by respond_node

print("State defined! Keys:", list(ConversationState.__annotations__.keys()))

### Step 2 — Write Node Functions

**Key rule:** Every node function:
- Takes `state` as input (the full dictionary)
- Returns a **dictionary** with only the keys it wants to update
- Does NOT need to return all keys — just the ones it changed

In [ ]:
from langchain_groq import ChatGroq
import os

# Re-initialize llm here so this section works even if you run it fresh
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

def greet_node(state: ConversationState) -> dict:
    """Node 1: Creates a greeting from the user's name."""
    name = state["user_name"]
    greeting = f"Hello, {name}! Welcome to LangGraph."
    print(f"  [greet_node] Running... greeting = '{greeting}'")
    return {"greeting": greeting}  # Only returns what it changed


def process_node(state: ConversationState) -> dict:
    """Node 2: Processes the greeting — adds some info."""
    greeting = state["greeting"]  # Reads what greet_node wrote
    processed = greeting + " Today we're learning about stateful workflows."
    print(f"  [process_node] Running... added context")
    return {"processed": processed}


def respond_node(state: ConversationState) -> dict:
    """Node 3: Uses the LLM to generate a final enthusiastic response."""
    processed = state["processed"]  # Reads what process_node wrote
    prompt = f"Given this message: '{processed}' — write an enthusiastic 1-sentence reply."
    response = llm.invoke(prompt)
    print(f"  [respond_node] Running... LLM called")
    return {"final_response": response.content}


print("All 3 node functions defined!")

### Step 3 — Build the Graph

In [ ]:
from langgraph.graph import StateGraph, END

# Create a graph that uses ConversationState
graph = StateGraph(ConversationState)

# Add nodes — format: add_node("node_name", function)
graph.add_node("greet",   greet_node)
graph.add_node("process", process_node)
graph.add_node("respond", respond_node)

# Set the entry point — where does the graph START?
graph.set_entry_point("greet")

# Add edges — the flow between nodes
graph.add_edge("greet",   "process")  # after greet → go to process
graph.add_edge("process", "respond")  # after process → go to respond
graph.add_edge("respond", END)        # after respond → DONE

print("Graph built! Nodes:", ["greet", "process", "respond"])
print("Flow: greet → process → respond → END")

### Step 4 — Compile and Run

In [ ]:
# Compile turns the graph definition into a runnable app
app = graph.compile()

print("Running the graph...")
print("-" * 40)

# Invoke with initial state — only provide the starting keys
result = app.invoke({"user_name": "Kiran"})

print("-" * 40)
print("\n=== Final State ===")
for key, value in result.items():
    print(f"  {key}: {value}")

**What just happened?**
1. The graph started at `greet_node` with `{"user_name": "Kiran"}`
2. `greet_node` added `greeting` to state
3. `process_node` read `greeting`, added `processed`
4. `respond_node` read `processed`, called the LLM, added `final_response`
5. The graph hit `END` and returned the full final state

Each node only saw one function. LangGraph handled passing state between them.

---
## 2. Conditional Edges — The Agent Decides Which Path

Now let's add **decision making**. The agent reads a question and decides:
- `"simple"` → answer directly from LLM knowledge
- `"needs_search"` → would need web search (we'll simulate this)

```
              ┌→ [direct_answer_node] ──┐
[router_node] ┤                         ├→ [END]
              └→ [search_answer_node]  ──┘
```

In [ ]:
# llm is initialized in Setup (Section 0). If you get NameError, run Section 0 first.
# Or uncomment the next two lines:
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

# ── State ──────────────────────────────────────────────────
class RouterState(TypedDict):
    question: str
    route: str       # "simple" or "needs_search"
    answer: str


# ── Node 1: Router — reads question, decides which path ────
def router_node(state: RouterState) -> dict:
    """Reads the question and decides: simple or needs_search."""
    question = state["question"]
    
    prompt = f"""Look at this question: '{question}'
    
Does answering it require current/live information (like news, prices, sports scores, 
recent events)? Or can it be answered from general knowledge?

Reply with ONLY one word: 'needs_search' or 'simple'"""
    
    response = llm.invoke(prompt)
    route = response.content.strip().lower()
    
    # Clean up — sometimes the LLM adds extra words
    if "search" in route:
        route = "needs_search"
    else:
        route = "simple"
    
    print(f"  [router_node] Question: '{question[:50]}...'")
    print(f"  [router_node] Decision: {route}")
    return {"route": route}


# ── Node 2a: Direct answer (no search needed) ─────────────
def direct_answer_node(state: RouterState) -> dict:
    """Answers the question directly using LLM knowledge."""
    question = state["question"]
    response = llm.invoke(f"Answer this question concisely: {question}")
    print(f"  [direct_answer_node] Answered from LLM knowledge")
    return {"answer": response.content}


# ── Node 2b: Search answer (would use real search) ────────
def search_answer_node(state: RouterState) -> dict:
    """Simulates a web search then answers."""
    question = state["question"]
    # In a real system: call DuckDuckGoSearchTool() here
    # For this demo we simulate a search result
    simulated_search = f"[Simulated search result for: {question}]"
    response = llm.invoke(
        f"Using this search result: '{simulated_search}' — answer: {question}. "
        f"Note that this is simulated data for demo purposes."
    )
    print(f"  [search_answer_node] Answered using (simulated) search")
    return {"answer": response.content}


# ── Conditional routing function ──────────────────────────
# This function reads state and returns the NAME of the next node
def decide_route(state: RouterState) -> str:
    route = state["route"]
    if route == "needs_search":
        return "search_answer"   # Must match the node name below
    else:
        return "direct_answer"   # Must match the node name below


# ── Build the graph ────────────────────────────────────────
router_graph = StateGraph(RouterState)

router_graph.add_node("router",        router_node)
router_graph.add_node("direct_answer", direct_answer_node)
router_graph.add_node("search_answer", search_answer_node)

router_graph.set_entry_point("router")

# After router → run decide_route to pick which node comes next
router_graph.add_conditional_edges(
    "router",         # FROM this node
    decide_route,     # CALL this function to decide WHERE to go
    {
        "direct_answer": "direct_answer",  # if fn returns "direct_answer" → go here
        "search_answer": "search_answer",  # if fn returns "search_answer" → go here
    }
)

router_graph.add_edge("direct_answer", END)
router_graph.add_edge("search_answer", END)

router_app = router_graph.compile()
print("Router graph compiled!")

In [ ]:
# Test with a simple question — should go to direct_answer
print("=" * 50)
print("Question 1: General knowledge")
print("=" * 50)
result1 = router_app.invoke({"question": "What is the capital of India?"})
print(f"\nAnswer: {result1['answer']}")

In [ ]:
# Test with a question needing current info — should go to search_answer
print("=" * 50)
print("Question 2: Needs current information")
print("=" * 50)
result2 = router_app.invoke({"question": "Who won the IPL 2025 final?"})
print(f"\nRoute taken: {result2['route']}")
print(f"Answer: {result2['answer']}")

In [ ]:
# Test a third question — see which path it takes
print("=" * 50)
print("Question 3: You decide the route!")
print("=" * 50)
result3 = router_app.invoke({"question": "What is machine learning?"})
print(f"\nRoute taken: {result3['route']}")
print(f"Answer: {result3['answer']}")

**Key insight:** Same graph, same code, 3 different questions → different paths chosen automatically. The LLM inside `router_node` makes the routing decision. You didn't hardcode any rules.

---
## 3. Loops — A Review Agent That Retries Until Quality Is Met

This is where LangGraph really shines. We'll build:

```
[draft] → [review] → PASS → [publish] → END
                   ↘ FAIL → [draft]   (loops back!)
```

The agent writes a draft, reviews it, and if the quality isn't good enough, it rewrites and tries again. This loop continues until the review passes.

In [ ]:
# llm is initialized in Setup (Section 0). If you get NameError, run Section 0 first.
# Or uncomment the next two lines:
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

from typing import TypedDict
from langgraph.graph import StateGraph, END

# ── State ──────────────────────────────────────────────────
class ReviewState(TypedDict):
    topic: str
    draft: str
    review_result: str   # "PASS" or "FAIL: <reason>"
    revision_count: int  # Track how many times we've revised
    final_post: str


# ── Node 1: Draft writer ───────────────────────────────────
def draft_node(state: ReviewState) -> dict:
    """Writes (or rewrites) a blog post draft."""
    topic = state["topic"]
    revision_count = state.get("revision_count", 0)
    previous_review = state.get("review_result", "")
    
    if revision_count == 0:
        # First attempt
        prompt = f"Write a short blog post (3-4 sentences) about: {topic}"
        print(f"  [draft_node] Writing first draft...")
    else:
        # Revision — use feedback from review
        previous_draft = state.get("draft", "")
        prompt = f"""Rewrite this draft to fix the issue: {previous_review}
        
Original draft: {previous_draft}

Write an improved version (3-4 sentences)."""
        print(f"  [draft_node] Rewriting draft (attempt #{revision_count + 1})...")
    
    response = llm.invoke(prompt)
    return {
        "draft": response.content,
        "revision_count": revision_count + 1
    }


# ── Node 2: Quality reviewer ───────────────────────────────
def review_node(state: ReviewState) -> dict:
    """Reviews the draft. Returns PASS or FAIL with a reason."""
    draft = state["draft"]
    
    prompt = f"""Review this blog post draft:

{draft}

Check: Does it have a clear opening? Does it give at least 2 specific facts or details?
Does it end with a conclusion?

Reply with ONLY:
- 'PASS' if all three criteria are met
- 'FAIL: <specific reason>' if any criterion is missing"""
    
    response = llm.invoke(prompt)
    result = response.content.strip()
    print(f"  [review_node] Review result: {result[:60]}")
    return {"review_result": result}


# ── Node 3: Publisher ──────────────────────────────────────
def publish_node(state: ReviewState) -> dict:
    """Publishes the approved draft."""
    print(f"  [publish_node] ✅ Draft approved! Publishing...")
    return {"final_post": state["draft"]}


# ── Conditional routing function ──────────────────────────
def route_after_review(state: ReviewState) -> str:
    review = state["review_result"]
    revision_count = state.get("revision_count", 0)
    
    # Safety valve: after 3 attempts, publish anyway
    if revision_count >= 3:
        print(f"  [route] Max revisions reached — publishing anyway")
        return "publish"
    
    if review.upper().startswith("PASS"):
        return "publish"
    else:
        return "draft"   # LOOP BACK to draft!


# ── Build the graph ────────────────────────────────────────
review_graph = StateGraph(ReviewState)

review_graph.add_node("draft",   draft_node)
review_graph.add_node("review",  review_node)
review_graph.add_node("publish", publish_node)

review_graph.set_entry_point("draft")
review_graph.add_edge("draft", "review")

review_graph.add_conditional_edges(
    "review",
    route_after_review,
    {
        "draft":   "draft",    # FAIL → loop back to draft
        "publish": "publish",  # PASS → go to publish
    }
)

review_graph.add_edge("publish", END)

review_app = review_graph.compile()
print("Review loop graph compiled!")

In [ ]:
print("Starting review loop for topic: 'The benefits of AI in healthcare'")
print("=" * 55)

result = review_app.invoke({"topic": "The benefits of AI in healthcare"})

print("=" * 55)
print(f"\nTotal revisions made: {result['revision_count']}")
print(f"Final review result: {result['review_result']}")
print(f"\n=== Published Post ===")
print(result['final_post'])

**What made this work:**
- The conditional edge after `review` can return `"draft"` — the name of a node that already ran
- LangGraph does not care if you loop back. It just follows the returned node name
- The `revision_count` in state tracks how many times we've looped
- The safety valve at 3 attempts prevents infinite loops

This is the **self-correction pattern** — used in production AI systems everywhere.

---
## 4. Human-in-the-Loop — Pause, Review, Resume

Now let's pause the workflow and wait for a human (you!) to approve before publishing.

**Two new things you need:**
1. `MemorySaver` — saves the state so the graph can be resumed
2. `interrupt_before=["publish"]` — tells the graph to stop BEFORE running `publish_node`

In [ ]:
# llm is initialized in Setup (Section 0). If you get NameError, run Section 0 first.
# Or uncomment the next two lines:
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
from typing import TypedDict

# ── State ──────────────────────────────────────────────────
class HITLState(TypedDict):
    topic: str
    draft: str
    human_approved: bool
    final_post: str


# ── Nodes ──────────────────────────────────────────────────
def write_draft(state: HITLState) -> dict:
    topic = state["topic"]
    response = llm.invoke(f"Write a 3-sentence blog post about: {topic}")
    print(f"  [write_draft] Draft written")
    return {"draft": response.content}


def publish_post(state: HITLState) -> dict:
    # This node only runs AFTER human approval
    print(f"  [publish_post] ✅ Human approved! Publishing...")
    return {"final_post": state["draft"], "human_approved": True}


# ── Build the graph — same as before ──────────────────────
hitl_graph = StateGraph(HITLState)
hitl_graph.add_node("write_draft",  write_draft)
hitl_graph.add_node("publish_post", publish_post)

hitl_graph.set_entry_point("write_draft")
hitl_graph.add_edge("write_draft",  "publish_post")
hitl_graph.add_edge("publish_post", END)

# ── The two HITL ingredients ───────────────────────────────
# 1. Create a checkpointer (saves state between pauses)
checkpointer = MemorySaver()

# 2. Compile with interrupt_before — pause BEFORE publish_post runs
hitl_app = hitl_graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["publish_post"]   # <-- pause here
)

print("HITL graph compiled!")
print("This graph will PAUSE before publishing and wait for your approval.")

In [ ]:
# Every HITL run needs a unique thread_id
# This is how LangGraph knows WHICH paused workflow to resume
config = {"configurable": {"thread_id": "blog-run-001"}}

print("=" * 50)
print("STEP 1: Starting workflow...")
print("=" * 50)

# First invoke — runs write_draft, then PAUSES before publish_post
hitl_app.invoke(
    {"topic": "Why Python is the best language for AI"},
    config=config
)

print("\n⏸️  WORKFLOW PAUSED — waiting for human approval")

In [ ]:
# Read the current state while paused
# This is what a human would review
current_state = hitl_app.get_state(config)

print("=" * 50)
print("STEP 2: Reading the draft for human review")
print("=" * 50)
draft = current_state.values.get("draft", "")
print(f"\n📄 Draft to review:\n")
print(draft)
print(f"\n⏸️  Workflow is paused. Next node would be: {current_state.next}")

In [ ]:
# STEP 3: Human makes a decision
# In a real app, this would be a UI button or input form

human_decision = input("\nDo you approve this draft? Type 'yes' to publish, 'no' to skip: ")

if human_decision.lower() == "yes":
    print("\n=" * 50)
    print("STEP 4: Resuming workflow with approval...")
    print("=" * 50)
    
    # Resume by calling invoke again with the SAME config
    # Pass None as input — the state is already saved from before
    final_result = hitl_app.invoke(None, config=config)
    
    print("\n✅ Published! Final post:")
    print(final_result["final_post"])
else:
    print("\n❌ Skipped publishing. The workflow remains paused.")
    print("You can resume it later with: hitl_app.invoke(None, config=config)")

**The magic explained:**
- `MemorySaver` saved the entire state (including the draft) when the graph paused
- `hitl_app.get_state(config)` let us read the saved state without resuming
- `hitl_app.invoke(None, config=config)` resumed — `None` means "use the saved state"
- The same `thread_id` in config tells LangGraph which saved run to resume

In production, the human review step could be a web UI, a Slack message, an email — anything. LangGraph just handles the pause/resume.

---
## 5. 🏆 Supervisor Pattern — One Router, Three Specialist Agents

The most powerful pattern in LangGraph. One **supervisor** reads a task and delegates to the right specialist:

```
                ┌→ [research_agent] ──┐
[supervisor] ───┼→ [code_agent]     ──┼→ [supervisor] → FINISH → END
                └→ [writer_agent]   ──┘
```

After each specialist finishes, control returns to the supervisor, which decides: call another specialist or finish.

In [ ]:
# llm is initialized in Setup (Section 0). If you get NameError, run Section 0 first.
# Or uncomment the next two lines:
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

from typing import TypedDict, List
from langgraph.graph import StateGraph, END
import json

# ── State ──────────────────────────────────────────────────
class SupervisorState(TypedDict):
    task: str                # The user's original task
    next_agent: str          # Which agent to call next ("research", "code", "writer", "FINISH")
    research_output: str     # What research_agent found
    code_output: str         # What code_agent wrote
    writer_output: str       # What writer_agent produced
    final_answer: str        # Assembled final answer
    steps_taken: List[str]   # Track which agents ran


# ── Supervisor node ────────────────────────────────────────
def supervisor_node(state: SupervisorState) -> dict:
    """Reads the task + what's been done so far, decides what to do next."""
    task = state["task"]
    steps_taken = state.get("steps_taken", [])
    
    # Build context of what's already been done
    done_summary = ", ".join(steps_taken) if steps_taken else "nothing yet"
    
    prompt = f"""You are a supervisor managing three specialist agents:
- research: looks up information and facts  
- code: writes Python code examples
- writer: writes summaries and explanations

Task: {task}
Already completed: {done_summary}

What should happen next? Choose ONE:
- 'research' — if we need information first
- 'code' — if we need a code example
- 'writer' — if we need a written explanation or summary
- 'FINISH' — if the task is fully done

Reply with ONLY the single word choice."""
    
    response = llm.invoke(prompt)
    decision = response.content.strip().lower()
    
    # Clean the decision
    for option in ["research", "code", "writer", "finish"]:
        if option in decision:
            decision = option
            break
    else:
        decision = "FINISH"  # Default if something unexpected
    
    print(f"  [supervisor] Steps done: {done_summary}")
    print(f"  [supervisor] Next: {decision}")
    return {"next_agent": decision}


# ── Specialist nodes ───────────────────────────────────────
def research_agent(state: SupervisorState) -> dict:
    """Looks up information relevant to the task."""
    task = state["task"]
    response = llm.invoke(
        f"Research and provide 3 key facts about this topic: {task}. "
        f"Be specific and factual. Format as a numbered list."
    )
    print(f"  [research_agent] Research complete")
    steps = state.get("steps_taken", []) + ["research"]
    return {"research_output": response.content, "steps_taken": steps}


def code_agent(state: SupervisorState) -> dict:
    """Writes a Python code example for the task."""
    task = state["task"]
    research = state.get("research_output", "")
    context = f"Based on this research: {research[:300]}" if research else ""
    response = llm.invoke(
        f"Write a short, clear Python code example for: {task}. "
        f"{context}. Include comments. Keep it under 15 lines."
    )
    print(f"  [code_agent] Code written")
    steps = state.get("steps_taken", []) + ["code"]
    return {"code_output": response.content, "steps_taken": steps}


def writer_agent(state: SupervisorState) -> dict:
    """Writes a clear explanation combining research and code."""
    task = state["task"]
    research = state.get("research_output", "")
    code = state.get("code_output", "")
    
    prompt = f"""Write a clear, beginner-friendly explanation for: {task}
    
Research gathered: {research[:400] if research else 'None'}
Code example: {code[:400] if code else 'None'}

Write 2-3 paragraphs that explain the topic simply."""
    
    response = llm.invoke(prompt)
    print(f"  [writer_agent] Explanation written")
    steps = state.get("steps_taken", []) + ["writer"]
    return {"writer_output": response.content, "steps_taken": steps}


# ── Final assembly node ────────────────────────────────────
def assemble_output(state: SupervisorState) -> dict:
    """Combines all agent outputs into a final answer."""
    parts = []
    if state.get("research_output"):
        parts.append(f"### Key Facts\n{state['research_output']}")
    if state.get("code_output"):
        parts.append(f"### Code Example\n{state['code_output']}")
    if state.get("writer_output"):
        parts.append(f"### Explanation\n{state['writer_output']}")
    
    final = "\n\n".join(parts) if parts else "No output generated."
    print(f"  [assemble_output] Final answer assembled")
    return {"final_answer": final}


# ── Routing function ───────────────────────────────────────
def route_to_agent(state: SupervisorState) -> str:
    next_agent = state["next_agent"]
    if next_agent == "finish":
        return "assemble"
    return next_agent  # returns "research", "code", or "writer"


# ── Build the graph ────────────────────────────────────────
sup_graph = StateGraph(SupervisorState)

# Add all nodes
sup_graph.add_node("supervisor",     supervisor_node)
sup_graph.add_node("research",       research_agent)
sup_graph.add_node("code",           code_agent)
sup_graph.add_node("writer",         writer_agent)
sup_graph.add_node("assemble",       assemble_output)

# Start at supervisor
sup_graph.set_entry_point("supervisor")

# Supervisor decides where to go
sup_graph.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {
        "research": "research",
        "code":     "code",
        "writer":   "writer",
        "assemble": "assemble",
    }
)

# After each specialist → return to supervisor
sup_graph.add_edge("research", "supervisor")
sup_graph.add_edge("code",     "supervisor")
sup_graph.add_edge("writer",   "supervisor")
sup_graph.add_edge("assemble", END)

supervisor_app = sup_graph.compile()
print("Supervisor graph compiled! Ready to run.")

In [ ]:
# Task 1 — needs all three agents
print("=" * 55)
print("Task: Explain Python list comprehensions with a code example")
print("=" * 55)

result = supervisor_app.invoke({
    "task": "Explain Python list comprehensions with a code example",
    "steps_taken": []
})

print(f"\nAgents used in order: {result['steps_taken']}")
print("\n" + "=" * 55)
print("FINAL ANSWER")
print("=" * 55)
print(result["final_answer"])

In [ ]:
# Task 2 — try a different task, see which agents it uses
print("=" * 55)
print("Task: What is the difference between RAM and ROM?")
print("=" * 55)

result2 = supervisor_app.invoke({
    "task": "What is the difference between RAM and ROM?",
    "steps_taken": []
})

print(f"\nAgents used: {result2['steps_taken']}")
print("\n" + "=" * 55)
print("FINAL ANSWER")
print("=" * 55)
print(result2["final_answer"])

---
## 6. 🎯 Your Turn — Challenge Exercises

### Challenge 1 (Easy): Add a fourth specialist to the supervisor

Add a `summariser_agent` that takes all the outputs and creates a 1-sentence summary. Add it as a new node, connect it back to supervisor, and test it.

In [ ]:
# YOUR CODE HERE
# Hint: Copy the supervisor graph from Section 5
# Add a new node: summariser_agent
# It should read all outputs and return a 1-sentence summary
# Add it to the routing function and connect it back to supervisor

### Challenge 2 (Medium): Make the review loop strict

Modify the review loop from Section 3 so the reviewer checks for a specific quality:
- The draft MUST mention at least one statistic or number
- If it doesn't, the review returns FAIL and asks the drafter to add one
- Test with a topic like "The importance of sleep"

In [ ]:
# YOUR CODE HERE
# Hint: Change the review prompt in review_node to check for numbers/statistics
# The draft_node already uses feedback from review_result — just change the review criteria

### Challenge 3 (Hard): Combine HITL + Supervisor

Build a supervisor that:
1. Calls research_agent and writer_agent
2. Then PAUSES for human approval before assembling the final answer
3. Only assembles if the human approves

Hint: You'll need `MemorySaver` and `interrupt_before=["assemble"]`

In [ ]:
# YOUR CODE HERE
# This combines Sections 4 and 5
# Use checkpointer=MemorySaver() and interrupt_before=["assemble"] in compile()

---
## 7. Day 8 Wrap-up

### What you built today:
- ✅ **Hello LangGraph** — 3-node graph, understood state flow, compile, invoke
- ✅ **Conditional Router** — LLM decides between paths automatically
- ✅ **Looping Review Agent** — drafts until quality is met (self-correction)
- ✅ **Human-in-the-Loop** — pause, review, resume with MemorySaver
- ✅ **Supervisor Pattern** — multi-agent system with automatic task delegation

### The 4-Step Pattern (you'll use this forever):
```python
# 1. Define State
class MyState(TypedDict): ...

# 2. Write node functions
def my_node(state): return {"key": value}

# 3. Build the graph
graph = StateGraph(MyState)
graph.add_node(...)
graph.add_edge(...)
graph.add_conditional_edges(...)

# 4. Compile and run
app = graph.compile()
result = app.invoke({...})
```

### Tomorrow — Day 9: Autonomous Agents & Self-Reflection
Agents that evaluate their own output, reflect on mistakes, and improve — without you telling them to.

---
### 📚 Resources to explore:
- [LangGraph official docs](https://langchain-ai.github.io/langgraph/)
- [LangGraph tutorials](https://langchain-ai.github.io/langgraph/tutorials/)
- [Human-in-the-loop guide](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/)